# Stage 2 — Text Splitting

In this notebook we convert the normalized records created during
document ingestion into LangChain Documents and split them into
retrieval-friendly chunks.

Goals:
- Preserve document metadata and source traceability
- Use LangChain for text splitting
- Keep tables and structured rows intact
- Avoid splitting useful policy sections arbitrarily
- Produce chunks that can later be embedded and stored in a vector database

## Imports

In [2]:
import json
from pathlib import Path
from collections import Counter

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Libraries Imported!")

Libraries Imported!


## Define Paths

In [3]:
PROJECT_PATH = Path.cwd().parent

DATA_DIR = PROJECT_PATH / "data"
PROCESSED_DIR = DATA_DIR / "processed"

NORMALIZED_DIR = PROCESSED_DIR / "normalized"
CHUNKS_DIR = PROCESSED_DIR / "chunks"

NORMALIZED_FILE = NORMALIZED_DIR / "normalized_documents.jsonl"
CHUNKS_FILE = CHUNKS_DIR / "chunks.jsonl"

print("Normalized file:", NORMALIZED_FILE)
print("Chunks file:", CHUNKS_FILE)

Normalized file: /Users/pushkarkamat/Desktop/financial-rag/data/processed/normalized/normalized_documents.jsonl
Chunks file: /Users/pushkarkamat/Desktop/financial-rag/data/processed/chunks/chunks.jsonl


## Load Normalized Records

In [4]:
def load_jsonl(path: Path) -> list[dict]:
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            records.append(json.loads(line))

    return records


normalized_records = load_jsonl(NORMALIZED_FILE)

print(f"Loaded records: {len(normalized_records):,}")

Loaded records: 868


## Inspect Record Types

In [5]:
record_type_counts = Counter(
    record.get("content_type", "unknown")
    for record in normalized_records
)

record_type_counts

Counter({'text': 840, 'table': 28})

## Convert records to LangChain Documents

In [6]:
def record_to_langchain_document(record: dict) -> Document:
    metadata = {
        "document": record.get("document"),
        "file_type": record.get("file_type"),
        "page": record.get("page"),
        "sheet": record.get("sheet"),
        "row_start": record.get("row_start"),
        "row_end": record.get("row_end"),
        "content_type": record.get("content_type"),
        "section": record.get("section"),
        "image_path": record.get("image_path"),
        "source_path": record.get("source_path"),
        "table_id": record.get("table_id"),
    }

    return Document(
        page_content=record.get("text", ""),
        metadata=metadata
    )

### Convert all records

In [7]:
documents = [
    record_to_langchain_document(record)
        for record in normalized_records
]

print(f"Langchain Document: {len(documents):,}")

Langchain Document: 868


In [8]:
documents[0]

Document(metadata={'document': '01_Credit_Risk_Policy_CRP-001_v2.0.docx', 'file_type': 'docx', 'page': None, 'sheet': None, 'row_start': None, 'row_end': None, 'content_type': 'text', 'section': None, 'image_path': None, 'source_path': '/Users/pushkarkamat/Desktop/financial-rag/data/raw/docx/01_Credit_Risk_Policy_CRP-001_v2.0.docx', 'table_id': None}, page_content='NORTHSTAR FINANCIAL')

## Create LangChain splitter

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

## Group related text records

The document ingestion stage intentionally preserves paragraphs, headings,
and tables as separate normalized records.

Before applying character-based splitting, related text records are grouped
into larger logical blocks. Tables and structured records remain independent.

This prevents very short paragraphs from becoming individual embedding chunks.

In [15]:
def group_text_records(documents: list[Document]) -> list[Document]:
    grouped_documents = []

    text_buffer = []
    buffer_metadata = None

    def flush_text_buffer():
        nonlocal text_buffer, buffer_metadata

        if not text_buffer:
            return

        combined_text = "\n\n".join(
            text.strip()
            for text in text_buffer
            if text.strip()
        )

        if combined_text:
            grouped_documents.append(
                Document(
                    page_content=combined_text,
                    metadata=buffer_metadata.copy()
                )
            )

        text_buffer = []
        buffer_metadata = None

    for document in documents:

        content_type = document.metadata.get("content_type")

        # Group text records.
        if content_type == "text":

            current_section = document.metadata.get("section")

            # Start a new text group.
            if not text_buffer:
                text_buffer.append(document.page_content)
                buffer_metadata = document.metadata.copy()
                continue

            previous_section = buffer_metadata.get("section")

            # Continue grouping if the section is the same.
            if current_section == previous_section:
                text_buffer.append(document.page_content)

            else:
                flush_text_buffer()

                text_buffer.append(document.page_content)
                buffer_metadata = document.metadata.copy()

        else:
            # Tables and other record types remain independent.
            flush_text_buffer()
            grouped_documents.append(document)

    # Flush anything remaining.
    flush_text_buffer()

    return grouped_documents

In [16]:
grouped_documents = group_text_records(documents)

print(f"Original records: {len(documents):,}")
print(f"Grouped records: {len(grouped_documents):,}")

Original records: 868
Grouped records: 218


## Split Grouped Documents into Chunks

In [21]:
def split_grouped_document(document: Document) -> list[Document]:
    content_type = document.metadata.get("content_type")

    # Normal policy text
    if content_type == "text":
        return text_splitter.split_documents([document])

    # Tables remain intact
    if content_type == "table":
        return [document]

    # Structured rows remain atomic
    if content_type == "structured_row":
        return [document]

    # Images are handled separately later
    if content_type == "image":
        return []

    # Fallback for any non-empty content
    if document.page_content.strip():
        return text_splitter.split_documents([document])

    return []

## Split grouped documents

In [ ]:
chunks = []

for document in grouped_documents:
    document_chunks = split_grouped_document(document)
    chunks.extend(document_chunks)

print(f"Grouped documents: {len(grouped_documents):,}")
print(f"Final chunks: {len(chunks):,}")

Grouped documents: 218
Final chunks: 295


## Assign Chunk Metadata

In [23]:
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"chunk_{index:08d}"
    chunk.metadata["char_count"] = len(chunk.page_content)

## Inspect Sample Chunks

In [ ]:
for i, chunk in enumerate(chunks[:10]):
    print(f"CHUNK {i}")
    print(f"ID: {chunk.metadata.get('chunk_id')}")
    print(f"Type: {chunk.metadata.get('content_type')}")
    print(f"Section: {chunk.metadata.get('section')}")
    print(f"Characters: {len(chunk.page_content)}")
    print()
    print(chunk.page_content)
    print("=" * 100)

## Save the chunks

In [25]:
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    for chunk in chunks:
        record = {
            "page_content": chunk.page_content,
            "metadata": chunk.metadata,
        }

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print(f"Saved {len(chunks):,} chunks to:")
print(CHUNKS_FILE)


Saved 295 chunks to:
/Users/pushkarkamat/Desktop/financial-rag/data/processed/chunks/chunks.jsonl


## Validate the saved file

In [26]:
saved_chunks = load_jsonl(CHUNKS_FILE)

print(f"Reloaded chunks: {len(saved_chunks):,}")
assert len(saved_chunks) == len(chunks)

print("Chunk file validation passed.")

Reloaded chunks: 295
Chunk file validation passed.
